# Reproducing a known result: recovering Kepler-8 b from raw photometry

**Goal:** independently re-derive the orbital period and planet/star radius ratio of a *known* transiting exoplanet, straight from Kepler light curves, and check our answers against the published values.

This is a deliberate warm-up. Before hunting for anything *new*, we want to prove the pipeline works on a signal whose right answer we already know. Recovering a known planet builds the validation instinct we'll need when a candidate is genuinely novel and there's no answer key.

**Target: Kepler-8 b** (`KIC 6922244`) — a hot Jupiter with a deep, unambiguous transit.

| Quantity | Published value |
|---|---|
| Orbital period | 3.52254 d |
| Rp/Rs (radius ratio) | ~0.094 |

Pipeline: `search → download → stitch → flatten → BLS periodogram → recover period → fold → measure depth → compare to literature`.

In [ ]:
import numpy as np
import lightkurve as lk
%matplotlib inline

TARGET = 'KIC 6922244'          # Kepler-8
PUBLISHED_PERIOD = 3.52254     # days
PUBLISHED_RP_RS = 0.094        # radius ratio

## 1. Find the data

Kepler observed in ~3-month "quarters". Each quarter is a separate data product; we search for all of the long-cadence (30-min) light curves for this star.

In [ ]:
search = lk.search_lightcurve(TARGET, author='Kepler', cadence='long')
print(f'{len(search)} light-curve products found')
search

## 2. Download and stitch

We take a handful of quarters (enough for a clean detection, few enough to download quickly), then `stitch()` them into one continuous light curve. Each quarter is normalized before stitching so quarter-to-quarter flux offsets don't create artificial jumps.

> Bump the slice up to `search[1:]` to use every quarter — it makes the folded transit crisper at the cost of a longer download.

In [ ]:
lcc = search[1:5].download_all(quality_bitmask='default')
lc = lcc.stitch().remove_nans()
lc.plot();

## 3. Flatten

The raw curve has slow trends (stellar variability, instrument drift) that swamp a ~1% transit. `flatten()` divides out a smooth trend with a Savitzky–Golay filter, leaving a curve that sits at 1.0 with the transits poking below it.

The `window_length` must be **longer than the transit** or the filter will treat the transit itself as trend and iron it flat. 901 cadences ≈ 19 days here — far longer than the ~2.4 h transit, so we're safe.

In [ ]:
flat = lc.flatten(window_length=901)
flat.plot();

## 4. Search for a period (Box Least Squares)

BLS slides a box-shaped dip across the data at thousands of trial periods and reports where a periodic dip best fits. The peak of the periodogram is our **independently recovered period** — we are *not* telling the algorithm the answer, only a plausible range (1–10 days).

In [ ]:
period_grid = np.linspace(1, 10, 20000)
bls = flat.to_periodogram(method='bls', period=period_grid, frequency_factor=500)
bls.plot();

rec_period = bls.period_at_max_power.value
t0 = bls.transit_time_at_max_power.value
dur = bls.duration_at_max_power.value
print(f'recovered period : {rec_period:.5f} d')
print(f'transit epoch t0 : {t0:.4f}')
print(f'transit duration : {dur*24:.2f} h')

### ✅ Validation check 1 — period

In [ ]:
err = abs(rec_period - PUBLISHED_PERIOD) / PUBLISHED_PERIOD * 100
print(f'recovered : {rec_period:.5f} d')
print(f'published : {PUBLISHED_PERIOD:.5f} d')
print(f'agreement : {err:.3f}% off')

## 5. Fold on the recovered period

If the period is right, folding every cycle on top of each other should stack all the transits into a single clean dip at phase 0. This is the visual proof the signal is real and periodic.

In [ ]:
folded = flat.fold(period=rec_period, epoch_time=t0)
folded.scatter(s=1);

## 6. Measure the transit depth → Rp/Rs

For a transit, depth ≈ (Rp/Rs)² — the fraction of starlight blocked is the ratio of the planet's disk area to the star's. So `sqrt(depth)` gives us the planet-to-star radius ratio, a second independent quantity to check.

We measure depth empirically: median flux out-of-transit minus median flux in-transit.

> **Gotcha worth internalizing:** a tempting earlier step is `remove_outliers()` to tidy the curve. Its default clips *low* outliers too — and a transit **is** a cluster of low outliers. Run it before this step and you'll shave the bottoms off your transits and *underestimate the depth by ~10×*. The period would still survive (enough points remain), but a physical measurement would be quietly wrong. Preprocessing that looks harmless can eat your signal; that's exactly the kind of self-inflicted error validation is meant to catch.

In [ ]:
phase = folded.phase.value
flux = folded.flux.value
half = (dur / 2) / rec_period

in_transit = np.abs(phase) < half
out_of_transit = (np.abs(phase) > 2 * half) & (np.abs(phase) < 0.25)

depth = np.nanmedian(flux[out_of_transit]) - np.nanmedian(flux[in_transit])
rp_rs = np.sqrt(depth)
print(f'transit depth : {depth*1e6:.0f} ppm')
print(f'implied Rp/Rs : {rp_rs:.4f}')

### ✅ Validation check 2 — radius ratio

In [ ]:
err = abs(rp_rs - PUBLISHED_RP_RS) / PUBLISHED_RP_RS * 100
print(f'recovered Rp/Rs : {rp_rs:.4f}')
print(f'published Rp/Rs : {PUBLISHED_RP_RS:.4f}')
print(f'agreement       : {err:.1f}% off')

## What we just did

Starting from nothing but raw pixel-derived light curves, we independently recovered **both** the orbital period (to ~0.005%) and the planet/star radius ratio (to a few %) of Kepler-8 b, and confirmed them against the literature. Two independent quantities agreeing with published values is strong evidence the pipeline is sound.

**Why this matters for discovery:** when we later point BLS + folding at a *candidate* with no answer key, we'll trust it precisely because it passed this test on a known object. We also learned a concrete failure mode — low-outlier clipping silently eating the signal — that we now know to watch for.

**Natural next steps:**
- Re-run on a *different* known planet (e.g. a shallower, longer-period one) to see where the pipeline gets harder.
- Cross-check the epoch `t0` and duration against the literature too.
- Then graduate to a *bounded novelty search*: run this same machinery over a small, well-defined sample and eyeball every candidate dip.